# Master — MOZAK: NOVE LEZIJE — EVALUACIJA

## Ćelija 0 — priprema, trener, metrike

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nnunetv2 nibabel scikit-learn

import os, glob, json, shutil, subprocess, csv
import numpy as np, nibabel as nib
from scipy.ndimage import label
import nnunetv2, torch

MASTER = '/content/drive/MyDrive/Master/mozak'
os.environ['nnUNet_raw']          = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'
for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed']]: os.makedirs(p, exist_ok=True)
REZ_DIR = f'{MASTER}/rezultati'; os.makedirs(REZ_DIR, exist_ok=True)
print('GPU:', torch.cuda.is_available())

TR_NAME = 'nnUNetTrainer_100epochs_save10'
vdir = os.path.join(os.path.dirname(nnunetv2.__file__), 'training', 'nnUNetTrainer', 'variants', 'training_length')
with open(os.path.join(vdir, f'{TR_NAME}.py'), 'w') as f:
    f.write("""import torch
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs
class nnUNetTrainer_100epochs_save10(nnUNetTrainer_100epochs):
    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 10
        self.initial_lr = 1e-3
        self.oversample_foreground_percent = 0.5
""")

STRUKT = np.ones((3,3,3), int)
def ucitaj(p): return np.asarray(nib.load(p).dataobj)

def dice(pred, gt):
    p = pred > 0; t = gt > 0
    if p.sum() + t.sum() == 0: return np.nan
    return float(2*(p & t).sum() / (p.sum() + t.sum()))

def lezijske(pred, gt):
    gl, ng = label(gt > 0, STRUKT); pl, npr = label(pred > 0, STRUKT)
    pm = pred > 0; gm = gt > 0
    tp = sum(1 for i in range(1, ng+1) if pm[gl == i].any())
    fp = sum(1 for j in range(1, npr+1) if not gm[pl == j].any())
    fn = ng - tp
    return dict(n_gt=ng, n_pred=npr, FP=fp,
                LTPR=tp/ng if ng else np.nan, LFPR=fp/npr if npr else 0.0,
                F1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan)

def red(ime, pred, gt, **extra):
    L = lezijske(pred, gt)
    return dict(pacijent=ime, Dice=dice(pred, gt), vox_pred=int((pred > 0).sum()), **L, **extra)

def ci95(vals, B=2000, seed=0):
    v = np.array([x for x in vals if x == x])
    if len(v) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = [rng.choice(v, len(v), replace=True).mean() for _ in range(B)]
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def protokol(rows, naziv, ref=None, stampaj=True):
    """ MSSEG-2: Dice samo za pacijente SA lezijama;
       za pacijente BEZ lezija broje se lazne detekcije."""
    les = [r for r in rows if r['n_gt'] > 0]; bez = [r for r in rows if r['n_gt'] == 0]
    out = dict(skup=naziv, n_sa=len(les), n_bez=len(bez))
    if les:
        d = [r['Dice'] for r in les]; lo, hi = ci95(d)
        out.update(Dice=float(np.nanmean(d)), CI_lo=lo, CI_hi=hi,
                   LTPR=float(np.nanmean([r['LTPR'] for r in les])),
                   LFPR=float(np.nanmean([r['LFPR'] for r in les])),
                   F1=float(np.nanmean([r['F1'] for r in les])))
    if bez:
        fp = [r['n_pred'] for r in bez]
        out.update(FP_bez=float(np.mean(fp)), bez_cisto=int(sum(1 for x in fp if x == 0)))
    if stampaj:
        print(f"\n=== {naziv} ===")
        if les:
            print(f"  SA lezijama (n={len(les)}): Dice={out['Dice']:.3f} [95% CI {out['CI_lo']:.3f}-{out['CI_hi']:.3f}]  "
                  f"LTPR={out['LTPR']:.3f}  LFPR={out['LFPR']:.3f}  F1={out['F1']:.3f}"
                  + (f"   | baseline: {ref}" if ref is not None else ""))
        if bez:
            print(f"  BEZ lezija (n={len(bez)}): prosecno laznih lezija={out['FP_bez']:.2f}; "
                  f"bez ijedne lazne: {out['bez_cisto']}/{len(bez)}")
    return out

def u_csv(rows, path):
    if not rows: return
    keys = list(rows[0].keys())
    for r in rows:
        for k in r:
            if k not in keys: keys.append(k)
    with open(path, 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader()
        for r in rows: w.writerow(r)
    print('sacuvano:', path)

print('spremno')

Mounted at /content/drive
GPU: True
spremno


## Ćelija 1

In [ ]:
from sklearn.model_selection import KFold

# MSSEG-2
MSSEG = None
for cand in ['/content/drive/MyDrive/training', '/content/drive/MyDrive/ms projekat/snimci_raspakovano/training']:
    if os.path.isdir(cand): MSSEG = cand; break
if MSSEG is None:
    h = glob.glob('/content/drive/MyDrive/**/flair_time01_on_middle_space.nii.gz', recursive=True)
    if h: MSSEG = os.path.dirname(os.path.dirname(h[0]))
assert MSSEG, 'MSSEG-2 nije nadjen'
cases = {}
for pid in sorted(os.listdir(MSSEG)):
    d = f'{MSSEG}/{pid}'
    t1 = f'{d}/flair_time01_on_middle_space.nii.gz'; t2 = f'{d}/flair_time02_on_middle_space.nii.gz'; m = f'{d}/ground_truth.nii.gz'
    if all(os.path.exists(x) for x in (t1, t2, m)): cases[f'msseg_{pid}'] = (t1, t2, m)
cids = sorted(cases)
print('MSSEG-2 pacijenata:', len(cids))

# open_ms longitudinalno (eksterno)
os.chdir('/content')
if not os.path.exists('/content/open_ms_data'):
    !git clone -q https://github.com/muschellij2/open_ms_data.git
oms_dir = '/content/open_ms_data/longitudinal/coregistered'
oms = {}
for pac in sorted(os.listdir(oms_dir)):
    d = f'{oms_dir}/{pac}'
    s1 = f'{d}/study1_FLAIR.nii.gz'; s2 = f'{d}/study2_FLAIR.nii.gz'; m = f'{d}/gt.nii.gz'
    if all(os.path.exists(x) for x in (s1, s2, m)): oms[f'oms_{pac}'] = (s1, s2, m)
print('open_ms pacijenata:', len(oms))

kf = KFold(n_splits=5, shuffle=True, random_state=12345)
SPLITS = [[cids[i] for i in te] for _, te in kf.split(cids)]

MODEL = f"{os.environ['nnUNet_results']}/Dataset021_BrainNew/{TR_NAME}__nnUNetPlans__3d_fullres"
FOLDS, CKPT = [], {}
for f in range(5):
    d = f'{MODEL}/fold_{f}'
    if os.path.exists(f'{d}/checkpoint_final.pth'):   FOLDS.append(f); CKPT[f] = 'checkpoint_final.pth'
    elif os.path.exists(f'{d}/checkpoint_best.pth'): FOLDS.append(f); CKPT[f] = 'checkpoint_best.pth'
    v = f'{d}/validation'
    if os.path.isdir(v):
        nn_val = sorted(x.replace('.nii.gz', '') for x in os.listdir(v) if x.endswith('.nii.gz'))
        print(f'fold {f}: podela se poklapa sa nnU-Net-om ->', nn_val == sorted(SPLITS[f]))
for f in FOLDS:
    print(f'fold {f}: {CKPT[f]}' + ('' if CKPT[f] == 'checkpoint_final.pth' else '   (trening NIJE zavrsen — privremeni rezultat)'))
assert FOLDS, 'Nema nijednog modela'
ENS = [f for f in FOLDS if CKPT[f] == 'checkpoint_final.pth'] or [FOLDS[0]]
ENS_CKPT = 'checkpoint_final.pth' if all(CKPT[f] == 'checkpoint_final.pth' for f in ENS) else CKPT[ENS[0]]
print('za eksterni test koristim foldove:', ENS, ENS_CKPT)

def predikcija(ulazi, out_dir, folds, chk):
    """ulazi: dict cid -> (kanal0, kanal1)"""
    in_dir = out_dir + '_in'
    shutil.rmtree(in_dir, ignore_errors=True); shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(in_dir)
    for cid, (a, b) in ulazi.items():
        shutil.copy(a, f'{in_dir}/{cid}_0000.nii.gz'); shutil.copy(b, f'{in_dir}/{cid}_0001.nii.gz')
    cmd = ['nnUNetv2_predict', '-i', in_dir, '-o', out_dir, '-d', '21', '-c', '3d_fullres',
           '-tr', TR_NAME, '-chk', chk, '-f'] + [str(f) for f in folds]
    subprocess.run(cmd, check=True)

MSSEG-2 pacijenata: 40
open_ms pacijenata: 20
fold 0: podela se poklapa sa nnU-Net-om -> True
fold 0: checkpoint_final.pth
za eksterni test koristim foldove: [0] checkpoint_final.pth


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('uredjaj:', DEVICE)

def predikcija(ulazi, out_dir, folds, chk):
    in_dir = out_dir + '_in'
    shutil.rmtree(in_dir, ignore_errors=True); shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(in_dir)
    for cid, (a, b) in ulazi.items():
        shutil.copy(a, f'{in_dir}/{cid}_0000.nii.gz'); shutil.copy(b, f'{in_dir}/{cid}_0001.nii.gz')
    cmd = ['nnUNetv2_predict', '-i', in_dir, '-o', out_dir, '-d', '21', '-c', '3d_fullres',
           '-tr', TR_NAME, '-chk', chk, '-device', DEVICE, '-f'] + [str(f) for f in folds]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError('predikcija nije uspela - poruka je iznad')
    print(f'predikcija gotova: {len(ulazi)} slučajeva')

uredjaj: cuda


## Ćelija 2 — MSSEG-2 (vs baseline 0.362)

In [ ]:
rows_cv = []
for f in FOLDS:
    out = f'/content/nl_cv/fold{f}'
    predikcija({cid: cases[cid][:2] for cid in SPLITS[f]}, out, [f], CKPT[f])
    for cid in SPLITS[f]:
        rows_cv.append(red(cid, ucitaj(f'{out}/{cid}.nii.gz'), ucitaj(cases[cid][2]) > 0, fold=f))
naziv_cv = (f'MSSEG-2 unakrsna validacija ({len(FOLDS)} foldova, {len(rows_cv)} pacijenata)'
            if len(FOLDS) > 1 else f'MSSEG-2 fold {FOLDS[0]} ({len(rows_cv)} pacijenata)')
R_CV = protokol(rows_cv, naziv_cv, ref=0.362)

predikcija gotova: 8 slučajeva

=== MSSEG-2 fold 0 (8 pacijenata) ===
  SA lezijama (n=5): Dice=0.660 [95% CI 0.421-0.843]  LTPR=0.867  LFPR=0.157  F1=0.821   | baseline: 0.362
  BEZ lezija (n=3): prosecno laznih lezija=0.33; bez ijedne lazne: 2/3


## Ćelija 3 — open_ms_data eksterno (vs baseline 0.280)

In [ ]:
out = '/content/nl_oms'
predikcija({cid: v[:2] for cid, v in oms.items()}, out, ENS, ENS_CKPT)
rows_oms = [red(cid, ucitaj(f'{out}/{cid}.nii.gz'), ucitaj(v[2]) > 0) for cid, v in oms.items()]
R_OMS = protokol(rows_oms, f'open_ms_data eksterno ({len(rows_oms)} pacijenata, foldovi {ENS})', ref=0.280)
print('\nNapomena: open_ms maske su PROMENE lezija (nove + uvecane), pa je ovaj test stroziji od cistih novih lezija.')

predikcija gotova: 20 slučajeva

=== open_ms_data eksterno (20 pacijenata, foldovi [0]) ===
  SA lezijama (n=20): Dice=0.304 [95% CI 0.206-0.402]  LTPR=0.163  LFPR=0.107  F1=0.258   | baseline: 0.28

Napomena: open_ms maske su PROMENE lezija (nove + uvecane), pa je ovaj test stroziji od cistih novih lezija.


## Ćelija 4 — provera: da li model zaista gleda RAZLIKU dva snimka?
Isti (noviji) snimak u oba kanala = „nema promene". Model koji uči razliku treba tada da prijavi **skoro ništa**.
Ako prijavljuje približno isto kao sa pravim parom → uči izgled lezije na novom snimku (ograničenje iz rada).

In [ ]:
out_abl = '/content/nl_abl'
predikcija({cid: (v[1], v[1]) for cid, v in oms.items()}, out_abl, ENS, ENS_CKPT)
rows_abl = []
for cid in oms:
    norm = ucitaj(f'/content/nl_oms/{cid}.nii.gz') > 0
    iden = ucitaj(f'{out_abl}/{cid}.nii.gz') > 0
    rows_abl.append(dict(pacijent=cid,
                         lezija_pravi_par=int(label(norm, STRUKT)[1]), vox_pravi_par=int(norm.sum()),
                         lezija_isti_snimak=int(label(iden, STRUKT)[1]), vox_isti_snimak=int(iden.sum())))
lp = np.mean([r['lezija_pravi_par'] for r in rows_abl]); li = np.mean([r['lezija_isti_snimak'] for r in rows_abl])
vp = np.mean([r['vox_pravi_par'] for r in rows_abl]);   vi = np.mean([r['vox_isti_snimak'] for r in rows_abl])
print(f'prosecno detekcija:  pravi par = {lp:.1f}   isti snimak 2x = {li:.1f}')
print(f'prosecno voksela:    pravi par = {vp:.0f}   isti snimak 2x = {vi:.0f}')
print(f'-> sa istim snimkom ostaje {100*vi/max(vp,1):.0f}% predikcije '
      '(blizu 0% = model koristi razliku; blizu 100% = gleda samo novi snimak)')

predikcija gotova: 20 slučajeva
prosecno detekcija:  pravi par = 8.3   isti snimak 2x = 0.1
prosecno voksela:    pravi par = 1060   isti snimak 2x = 0
-> sa istim snimkom ostaje 0% predikcije (blizu 0% = model koristi razliku; blizu 100% = gleda samo novi snimak)


## Ćelija 5 — čuvanje rezultata

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
u_csv(rows_cv,  f'{REZ_DIR}/nove_lezije_msseg2_po_pacijentu_{ts}.csv')
u_csv(rows_oms, f'{REZ_DIR}/nove_lezije_openms_po_pacijentu_{ts}.csv')
u_csv(rows_abl, f'{REZ_DIR}/nove_lezije_provera_razlike_{ts}.csv')
u_csv([R_CV, R_OMS], f'{REZ_DIR}/nove_lezije_SAZETAK_{ts}.csv')

sacuvano: /content/drive/MyDrive/Master/mozak/rezultati/nove_lezije_msseg2_po_pacijentu_20261110_1237.csv
sacuvano: /content/drive/MyDrive/Master/mozak/rezultati/nove_lezije_openms_po_pacijentu_20261110_1237.csv
sacuvano: /content/drive/MyDrive/Master/mozak/rezultati/nove_lezije_provera_razlike_20261110_1237.csv
sacuvano: /content/drive/MyDrive/Master/mozak/rezultati/nove_lezije_SAZETAK_20261110_1237.csv
